# LUMI 1단계: 단일 정지 이미지 내 광과민성(PSE) 위험 여부 판정 알고리즘 (`image_discriminator_2_adstripe.ipynb`)

본 노트북은 `image_discriminator_1_without_ccrp.ipynb`를 기반으로, **디바이스(`'laptop'` / `'desktop'`) 및 디스플레이 크기(`display_area`, inch)에 따른 표준 시청거리와 시야각($2\theta$)을 계산하여 공간 주파수(`cpd`, cycles per degree) 측정 기준($1.0 \sim 10.0\text{ cpd}$)을 정립**하고, 이를 통해 **멀리 떨어져 있는 별개의 물체들이 줄무늬로 합산되는 문제를 해결한 직선 줄무늬 탐지 버전**입니다.

### 핵심 추가 및 변경 사항
1. **[코드 셀 1-A] 표준 시청거리 가정을 통한 시야각 및 `cpd` 측정 기준 정립**
   - 입력 인자: `device: str = 'laptop'` (`'laptop'` 또는 `'desktop'`만 인식), `display_area: int = 15` (디스플레이 대각선 크기, inch)
   - 표준 시청거리($d$): `'laptop'` = **$56.0\text{ cm}$**, `'desktop'` = **$68.0\text{ cm}$** (`3694790.pdf` 17쪽 Table 2 기준, 화면비 16:9 전체화면 가정)
   - 시야각 공식(`3694790.pdf` 34쪽 Appendix A): $2\theta = 2\tan^{-1}\left(\frac{x}{2d}\right)$를 통해 화면 전체 가로·세로 시야각($^\circ$) 및 **시야각 $1^\circ$당 픽셀 수(`px_per_deg`)** 산출
   - 위험 공간 주파수 범위(**방식 B: $1.0 \sim 10.0\text{ cpd}$**, `1-s2.0-S0003687008001282-main.pdf` 505쪽 기준)에 대응하는 **줄무늬 1쌍(1 주기)의 최소·최대 허용 픽셀 간격** 정립
2. **[코드 셀 1-B] `cpd` 연속성 기준을 적용한 직선 줄무늬 패턴 판정 (방사형·동심원 미포함)**
   - 방사형 및 동심원 스캔을 제외하고 직선 줄무늬만 1픽셀 간격으로 정밀 스캔
   - 인접한 밝고 어두운 줄무늬 사이의 간격이 **$1.0 \le \text{cpd} \le 10.0$** 범위를 벗어나면(즉, 줄무늬가 멀리 떨어져 있으면 `$< 1.0\text{ cpd}$`, 혹은 눈의 해상도 한계를 넘으면 `$> 10.0\text{ cpd}$`) **연속된 줄무늬가 끊어진 것으로 판정하여 카운트를 즉시 리셋**하고, 끊기지 않고 연속된 최대 줄무늬 쌍 개수만 산출

### [코드 셀 1-A] 디바이스(`laptop`/`desktop`) · 디스플레이 크기(`inch`) 기반 표준 시청거리 · 시야각 · `cpd` 측정 기준 정립

In [ ]:
import os
import math
import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

# 분석할 단일 이미지 경로 및 기본 디스플레이 설정
IMAGE_PATH = r"이미지 경로"
#DEFAULT_DEVICE = "laptop"     
#DEFAULT_DISPLAY_AREA = 15     

@torch.no_grad()
def load_single_image_rgb(path: str, max_side: int = 1080) -> torch.Tensor:
    """단일 정지 이미지를 로드하여 GPU/CPU 텐서 (H, W, 3), float32 [0, 255] 형태로 반환합니다."""
    device_t = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if not os.path.exists(path):
        raise FileNotFoundError(f"입력 파일을 찾을 수 없습니다: {path}")

    buf = np.fromfile(path, dtype=np.uint8)
    frame_bgr = cv2.imdecode(buf, cv2.IMREAD_COLOR)
    if frame_bgr is None:
        raise ValueError(f"이미지를 읽을 수 없습니다: {path}")
    img_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)

    h, w = img_rgb.shape[:2]
    if max(h, w) > max_side:
        scale = max_side / float(max(h, w))
        img_rgb = cv2.resize(img_rgb, (int(w * scale), int(h * scale)), interpolation=cv2.INTER_AREA)
    return torch.from_numpy(img_rgb).to(device=device_t, dtype=torch.float32)

@torch.no_grad()
def srgb_to_linear_and_luminance(rgb_255: torch.Tensor):
    """방식 A (ITU-R BT.1702-3 / WCAG 2.2): sRGB [0,255] -> 선형 RGB [0,1] 및 물리 휘도 L [0, 200] cd/m^2"""
    
    c_s = torch.clamp(rgb_255 / 255.0, 0.0, 1.0) # sRGB Color
    """0~255를 0~1로 스케일링"""
    
    c_lin = torch.where(
        c_s <= 0.04045,
        c_s / 12.92,
        torch.pow((c_s + 0.055) / 1.055, 2.4)
    ) # Linear Color
    """
    사람의 눈은 밝은 곳보다 어두운 곳의 미세한 밝기 변화에 훨씬 민감함.
    제한된 숫자(0~255)에 어두운 색조를 촘촘하게 저장하기 위해 실제 빛의 에너지를 그대로 표현 x
    => 원래 sRGB는 감마 압축 (1/2.2)승 __ 여기선 2.4로 쓰임.
    픽셀값 0~10 수준에서는 미분값이 너무 낮음 -> 0.04045 이하에서는 직선으로 변환
    0.055, 1.055는 미분가능하게 만들기 위한 숫자
    """
    y_rel = 0.2126 * c_lin[..., 0] + 0.7152 * c_lin[..., 1] + 0.0722 * c_lin[..., 2] # 상대 휘도
    """ITU-R에서 정한 인간 시각 밝기 가중치"""
    lum_cd = 200.0 * y_rel # 물리 휘도
    """ITU-R에서 표준 일반 디스플레이(SDR)에서 최대 흰색(y_rel = 1.0)의 물리 휘도를 200cd/m^2로 기준 삼는다"""
    return c_lin, y_rel, lum_cd

def compute_viewing_angle_and_cpd_spec(
    img_w: int,
    img_h: int,
    device: str = "laptop",
    display_area: float = 15.6,
    cpd_min: float = 1.0,
    cpd_max: float = 10.0
) -> dict:
    """
    표준 시청거리 가정을 통해 디스플레이 시야각(Visual Angle)과 시야각 1도당 픽셀 수(px_per_deg),
    그리고 위험 공간 주파수(1.0 ~ 10.0 cpd) 판정을 위한 픽셀 간격 임계치를 정립합니다.
    - device: 'laptop' (표준 시청거리 56.0 cm) 또는 'desktop' (표준 시청거리 68.0 cm)
    - display_area: 디스플레이 대각선 크기 (inch, 기본값 15.6)
    - cpd_min, cpd_max는 되도록 건들지 말 것.
    =>필요하다면 cpd_min, cpd_max = 2, 8
    """
    if device == "laptop":
        viewing_dist_cm = 56.0 # 표준 시청 거리 넣은건데 이것도 입력으로 받아도 됨.
    elif device == "desktop":
        viewing_dist_cm = 68.0 # 표준 시청 거리 넣은건데 이것도 입력으로 받아도 됨.
    else:
        raise ValueError(f"device 인자는 'laptop' 또는 'desktop'만 허용됩니다. (입력값: {device!r})")

    if float(display_area) <= 0:
        raise ValueError(f"display_area(디스플레이 크기 inch)는 float여야 합니다. (입력값: {display_area})")

    # 1. 16:9 표준 화면비 기준 물리적 가로/세로 길이(cm) 계산 (1 inch = 2.54 cm)
    diag_cm = float(int(display_area)) * 2.54 # 화면에서 대각선 길이 입력을 인치로 받아서 cm로 바꿔줌
    aspect_diag = math.sqrt(16.0 ** 2 + 9.0 ** 2) # 16:9 화면비에서 가로, 세로 cm로 변환
    screen_w_cm = diag_cm * (16.0 / aspect_diag)
    screen_h_cm = diag_cm * (9.0 / aspect_diag)

    # 2. 시야각 계산: 2*theta = 2 * arctan(x / (2 * d)) (3694790.pdf Appendix A)
    fov_w_deg = 2.0 * math.degrees(math.atan(screen_w_cm / (2.0 * viewing_dist_cm))) # 전체화면 가로 시야각
    fov_h_deg = 2.0 * math.degrees(math.atan(screen_h_cm / (2.0 * viewing_dist_cm))) # 전체화면 세로 시야각

    # 3. 시야각 1도(1 degree)당 픽셀 수 및 위험 cpd(1.0 ~ 10.0 cpd)에 대응하는 줄무늬 1쌍(1주기) 픽셀 폭 범위
    px_per_deg_w = float(img_w) / fov_w_deg 
    px_per_deg_h = float(img_h) / fov_h_deg
    px_per_deg = 0.5 * (px_per_deg_w + px_per_deg_h) #### 판별할 때는 가로 세로 따로 하는데 잠깐만 이렇게 해놓음.

    # 1주기(밝은 줄 1개 + 어두운 줄 1개)의 허용 픽셀 길이: [px_per_deg / cpd_max, px_per_deg / cpd_min]
    min_cycle_px = px_per_deg / cpd_max   # 10.0 cpd 상한에 대응하는 최소 주기(px)
    max_cycle_px = px_per_deg / cpd_min   # 1.0 cpd 하한에 대응하는 최대 주기(px) -> 이보다 멀면 별개의 물체!

    return {
        "device": device,
        "display_area_inch": int(display_area),
        "viewing_dist_cm": viewing_dist_cm,
        "screen_w_cm": screen_w_cm,
        "screen_h_cm": screen_h_cm,
        "fov_w_deg": fov_w_deg,
        "fov_h_deg": fov_h_deg,
        "px_per_deg_w": px_per_deg_w,
        "px_per_deg_h": px_per_deg_h,
        "px_per_deg": px_per_deg,
        "cpd_min": cpd_min,
        "cpd_max": cpd_max,
        "min_cycle_px": min_cycle_px,
        "max_cycle_px": max_cycle_px,
        "min_half_cycle_px": min_cycle_px / 2.0,
        "max_half_cycle_px": max_cycle_px / 2.0,
    }

# 기준 정립 확인용 테이블 출력
_sample_rgb = load_single_image_rgb(IMAGE_PATH)
_sh, _sw = _sample_rgb.shape[:2]
_spec_info = compute_viewing_angle_and_cpd_spec(_sw, _sh)
df_cpd_spec = pd.DataFrame([{
    "기기 종류": _spec_info["device"],
    "화면 크기": f"{_spec_info['display_area_inch']} inch",
    "표준 시청 거리": f"{_spec_info['viewing_dist_cm']:.1f} cm",
    "디스플레이 시야각": f"{_spec_info['fov_w_deg']:.2f}° × {_spec_info['fov_h_deg']:.2f}°",
    "시야각 1°당 세로 픽셀 수":f"{_spec_info['px_per_deg_h']:.2f}", 
    "시야각 1°당 가로 픽셀 수": f"{_spec_info['px_per_deg_w']:.2f}",
    "위험 공간 주파수 범위": f"{_spec_info['cpd_min']:.1f} ~ {_spec_info['cpd_max']:.1f} cpd",
    "유효 줄무늬 1쌍 허용 간격": f"{_spec_info['min_cycle_px']:.2f} ~ {_spec_info['max_cycle_px']:.2f} px"
}], index=["CPD 측정 기준 설정값"])
display(df_cpd_spec)

,기기 종류,화면 크기,표준 시청 거리,디스플레이 시야각,시야각 1°당 세로 픽셀 수,시야각 1°당 가로 픽셀 수,위험 공간 주파수 범위,유효 줄무늬 1쌍 허용 간격
CPD 측정 기준 설정값,laptop,15 inch,56.0 cm,33.03° × 18.94°,31.79,32.70,1.0 ~ 10.0 cpd,3.22 ~ 32.24 px


### [코드 셀 1-B] `cpd` 기준을 적용한 직선 줄무늬 공간 패턴 판정 알고리즘 (방사형 미포함) — 면적 임계치 25%

In [14]:
def count_opposing_stripe_pairs_1d_cpd(
    profile: np.ndarray,
    px_per_deg: float,
    cpd_min: float = 1.0,
    cpd_max: float = 10.0
) -> tuple: # 1차원 어레이를 입력으로 받음
    """
    1차원 휘도 프로파일(1 px 간격 샘플링)에서:
    1) ITU-R BT.1702-3 명암 대비 기준(>=20 cd/m^2 또는 Michelson > 1/17)을 만족하고,
    2) 인접한 봉우리-골짜기 간격이 공간 주파수 1.0 ~ 10.0 cpd 범위 안에 들어오는
       '연속된(끊기지 않은) 줄무늬 쌍(contiguous light-dark pairs)'의 최대 개수와 평균 cpd를 반환합니다.
    * 줄무늬 사이가 멀리 떨어져 있어 cpd < 1.0 이 되면 연속 카운트를 즉시 0으로 리셋하여 합산을 방지합니다.
    """
    if profile.size < 4:
        return 0, 0.0

    # 1단계: (픽셀 위치 인덱스, 휘도 값)을 함께 기록하며 산봉우리와 골짜기 추출
    extrema_indices = [0]
    extrema_vals = [float(profile[0])]
    direction = 0

    for idx in range(1, profile.size):
        v = float(profile[idx])
        diff = v - extrema_vals[-1]
        if direction >= 0 and diff > 0:
            if direction == 0:
                extrema_vals.append(v)
                extrema_indices.append(idx)
            else:
                extrema_vals[-1] = v
                extrema_indices[-1] = idx
            direction = 1
        elif direction <= 0 and diff < 0:
            if direction == 0:
                extrema_vals.append(v)
                extrema_indices.append(idx)
            else:
                extrema_vals[-1] = v
                extrema_indices[-1] = idx
            direction = -1
        elif direction == 1 and (extrema_vals[-1] - v) >= 5.0: # 5로 할건지 다른 숫자로 할건지 정해야 함. 하이퍼파라미터 튜닝? 해야 함.
            extrema_vals.append(v)
            extrema_indices.append(idx)
            direction = -1
        elif direction == -1 and (v - extrema_vals[-1]) >= 5.0: # 5로 할건지 다른 숫자로 할건지 정해야 함. 하이퍼파라미터 튜닝? 해야 함.
            extrema_vals.append(v)
            extrema_indices.append(idx)
            direction = 1

    # 2단계: 명암 대비 기준 + 공간 주파수(1.0 <= cpd <= 10.0) 연속성 검사
    max_contiguous_transitions = 0
    best_run_cpds = []
    cur_transitions = 0
    cur_cpds = []

    for i in range(len(extrema_vals) - 1):
        l_high = max(extrema_vals[i], extrema_vals[i + 1])
        l_low = min(extrema_vals[i], extrema_vals[i + 1])
        cond_low = (l_low < 160.0) and ((l_high - l_low) >= 20.0)
        cond_high = (l_low >= 160.0) and (((l_high - l_low) / max(l_high + l_low, 1e-6)) > (1.0 / 17.0))
        contrast_ok = cond_low or cond_high

        # 봉우리~골짜기 반주기 거리(half_cycle_px)로부터 1주기(밝은 줄+어두운 줄 1쌍) 픽셀 폭 및 cpd 계산
        half_cycle_px = float(extrema_indices[i + 1] - extrema_indices[i])
        full_cycle_px = 2.0 * max(half_cycle_px, 1e-6)
        local_cpd = px_per_deg / full_cycle_px
        cpd_ok = (cpd_min <= local_cpd <= cpd_max)

        if contrast_ok and cpd_ok:
            """이렇게 안 하고 if 안에 if 안에 if 하면 안 되나... 
            contrast_ok false인데도 반복문 끝까지 가야 하는게 뭔가 효율성 떨어져 보임"""
            cur_transitions += 1
            cur_cpds.append(local_cpd)
            if cur_transitions > max_contiguous_transitions:
                max_contiguous_transitions = cur_transitions
                best_run_cpds = list(cur_cpds)
        else:
            # 대비가 부족하거나 줄무늬 간격이 너무 멀면(cpd < 1.0) 연속성이 끊어진 것으로 보고 즉시 리셋
            cur_transitions = 0
            cur_cpds = []

    max_pairs = max_contiguous_transitions // 2
    mean_cpd = float(np.mean(best_run_cpds)) if (max_pairs > 0 and len(best_run_cpds) > 0) else 0.0
    return max_pairs, mean_cpd
"""11-05 여기까지"""
@torch.no_grad()
def evaluate_spatial_pattern(
    image_path: str,
    area_threshold: float = 0.25,
    device: str = "laptop",
    display_area: int = 15
) -> pd.DataFrame:
    rgb = load_single_image_rgb(image_path)
    _, _, lum = srgb_to_linear_and_luminance(rgb)
    h, w = lum.shape

    # 0. 표준 시청거리 기반 시야각 및 cpd 기준 계산 (1.0 ~ 10.0 cpd)
    cpd_spec = compute_viewing_angle_and_cpd_spec(w, h, device=device, display_area=display_area, cpd_min=1.0, cpd_max=10.0)
    px_per_deg = cpd_spec["px_per_deg"]

    # 1. 국소 명암 대비(ITU-R BT.1702-3) 마스크 및 심각도 비율 계산
    k_size = max(7, int(min(h, w) * 0.03) | 1)
    pad = k_size // 2
    lum_4d = lum.unsqueeze(0).unsqueeze(0)
    l_high = F.max_pool2d(lum_4d, kernel_size=k_size, stride=1, padding=pad).squeeze()
    l_low = -F.max_pool2d(-lum_4d, kernel_size=k_size, stride=1, padding=pad).squeeze()

    delta_l = l_high - l_low
    michelson = delta_l / torch.clamp(l_high + l_low, min=1e-6)
    contrast_ratio = torch.where(l_low < 160.0, delta_l / 20.0, michelson / (1.0 / 17.0))
    contrast_mask = torch.where(l_low < 160.0, delta_l >= 20.0, michelson > (1.0 / 17.0))

    # 국소 공간 패턴 영역(고주파 에지/주기성 밀집 영역) 스무딩으로 패턴 점유 영역 산출
    env_k = max(15, int(min(h, w) * 0.06) | 1)
    pattern_density = F.avg_pool2d(contrast_mask.float().unsqueeze(0).unsqueeze(0), kernel_size=env_k, stride=1, padding=env_k // 2).squeeze()
    pattern_region_mask = pattern_density >= 0.15
    pattern_area_fraction = float(pattern_region_mask.float().mean().item())

    # 2. 직선 줄무늬 다방향 단면 스캔 (방사형·동심원 패턴 미포함, 1 px 등간격 샘플링으로 정확한 cpd 측정)
    lum_np = lum.detach().cpu().numpy()
    max_pairs = 0
    detected_cpd = 0.0

    cy, cx = h / 2.0, w / 2.0
    radius = int(min(h, w) * 0.45)
    t_vals = np.arange(-radius, radius + 1, 1.0)  # 1 step = 정확히 1 픽셀 거리

    for deg in range(0, 180, 15):
        rad = math.radians(deg)
        # 스캔 방향(각도 deg)에 따른 정확한 1도당 픽셀 수 보정
        fov_w_deg = cpd_spec["fov_w_deg"]
        fov_h_deg = cpd_spec["fov_h_deg"]
        px_per_deg_dir = 1.0 / math.sqrt((math.cos(rad) / (w / fov_w_deg)) ** 2 + (math.sin(rad) / (h / fov_h_deg)) ** 2)

        for offset in np.linspace(-radius * 0.5, radius * 0.5, 5):
            xs = np.clip(np.round(cx + t_vals * math.cos(rad) - offset * math.sin(rad)).astype(int), 0, w - 1)
            ys = np.clip(np.round(cy + t_vals * math.sin(rad) + offset * math.cos(rad)).astype(int), 0, h - 1)
            pairs, mean_cpd = count_opposing_stripe_pairs_1d_cpd(
                lum_np[ys, xs],
                px_per_deg=px_per_deg_dir,
                cpd_min=1.0,
                cpd_max=10.0
            )
            if pairs > max_pairs:
                max_pairs = pairs
                detected_cpd = mean_cpd

    # 3. 최종 위험/안전 판정 (1.0~10.0 cpd 내 연속 줄무늬 쌍 >= 5개 이고 패턴 면적 > 임계치)
    is_danger = (max_pairs >= 5) and (pattern_area_fraction > area_threshold)
    verdict = "위험 (DANGER)" if is_danger else "안전 (SAFE)"

    # 4. 판정 근거 확률 2가지 방식 계산
    mean_contrast_factor = float(torch.clamp(contrast_ratio[pattern_region_mask].mean() if pattern_region_mask.any() else contrast_ratio.max(), 0.0, 1.0).item())
    pair_factor = min(1.0, max_pairs / 5.0)
    eta = (pattern_area_fraction / area_threshold) * pair_factor * mean_contrast_factor
    prob_method_1 = (eta ** 2 / (1.0 + eta ** 2)) * 100.0
    prob_method_2 = (pattern_area_fraction * 100.0) if (max_pairs >= 5) else 0.0

    label_pct = int(round(area_threshold * 100))
    return pd.DataFrame([{
        "판정 결과": verdict,
        "판정 근거 확률 - 방식 1 (임계치 달성도 기반)": f"{prob_method_1:.2f}%",
        "판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)": f"{prob_method_2:.2f}%"
    }], index=[f"국소 공간 패턴 통합 (면적 기준 {label_pct}%, {device} {display_area}인치)"])

df_cell_1 = evaluate_spatial_pattern(
    IMAGE_PATH,
    area_threshold=0.25,
)
display(df_cell_1)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)
"국소 공간 패턴 통합 (면적 기준 25%, laptop 15인치)",위험 (DANGER),89.68%,73.69%


### [코드 셀 2] 국소 공간 패턴 통합(직선 줄무늬) 판정 알고리즘 — 면적 임계치 40%

In [3]:
# 분석할 단일 이미지 경로 (코드 셀 1과 동일하거나 개별 지정 가능)

@torch.no_grad()
def evaluate_spatial_pattern_40(
    image_path: str,
    device: str = "laptop",
    display_area: int = 15
) -> pd.DataFrame:
    df_res = evaluate_spatial_pattern(
        image_path,
        area_threshold=0.40,
        device=device,
        display_area=display_area
    )
    return df_res

df_cell_2 = evaluate_spatial_pattern_40(
    IMAGE_PATH,
    device=DEFAULT_DEVICE,
    display_area=DEFAULT_DISPLAY_AREA
)
display(df_cell_2)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)
"국소 공간 패턴 통합 (면적 기준 40%, laptop 15인치)",위험 (DANGER),77.24%,73.69%


### [코드 셀 3] 국소 고대비 영역 판정 알고리즘 — 면적 임계치 25%

In [4]:
@torch.no_grad()
def evaluate_high_contrast(image_path: str, area_threshold: float = 0.25) -> pd.DataFrame:
    """
    ITU-R BT.1702-3 명암 대비 기준 적용:
    - 어두운 영역 L_LOW < 160 cd/m^2 일 때: L_HIGH - L_LOW >= 20 cd/m^2
    - 어두운 영역 L_LOW >= 160 cd/m^2 일 때: Michelson 대비 (L_HIGH - L_LOW)/(L_HIGH + L_LOW) > 1/17
    """
    rgb = load_single_image_rgb(image_path)
    _, _, lum = srgb_to_linear_and_luminance(rgb)
    h, w = lum.shape

    # 국소 영역 내 최대/최소 휘도 산출
    k_size = max(9, int(min(h, w) * 0.04) | 1)
    pad = k_size // 2
    lum_4d = lum.unsqueeze(0).unsqueeze(0)
    l_high = F.max_pool2d(lum_4d, kernel_size=k_size, stride=1, padding=pad).squeeze()
    l_low = -F.max_pool2d(-lum_4d, kernel_size=k_size, stride=1, padding=pad).squeeze()

    delta_l = l_high - l_low
    michelson = delta_l / torch.clamp(l_high + l_low, min=1e-6)

    high_contrast_mask = torch.where(
        l_low < 160.0,
        delta_l >= 20.0,
        michelson > (1.0 / 17.0)
    )
    severity_ratio = torch.where(
        l_low < 160.0,
        delta_l / 20.0,
        michelson / (1.0 / 17.0)
    )

    contrast_area_fraction = float(high_contrast_mask.float().mean().item())
    is_danger = contrast_area_fraction > area_threshold
    verdict = "위험 (DANGER)" if is_danger else "안전 (SAFE)"

    # 방식 1, 2 확률 산출
    intensity_gate = float(torch.clamp(severity_ratio[high_contrast_mask].mean() if high_contrast_mask.any() else severity_ratio.max(), 0.0, 1.0).item())
    eta = (contrast_area_fraction / area_threshold) * intensity_gate
    prob_method_1 = (eta ** 2 / (1.0 + eta ** 2)) * 100.0
    prob_method_2 = contrast_area_fraction * 100.0

    label_pct = int(round(area_threshold * 100))
    return pd.DataFrame([{
        "판정 결과": verdict,
        "판정 근거 확률 - 방식 1 (임계치 달성도 기반)": f"{prob_method_1:.2f}%",
        "판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)": f"{prob_method_2:.2f}%"
    }], index=[f"국소 고대비 영역 (면적 기준 {label_pct}%)"])

df_cell_3 = evaluate_high_contrast(IMAGE_PATH, area_threshold=0.25)
display(df_cell_3)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)
국소 고대비 영역 (면적 기준 25%),위험 (DANGER),87.11%,65.00%


### [코드 셀 4] 국소 고대비 영역 판정 알고리즘 — 면적 임계치 40%

In [5]:
df_cell_4 = evaluate_high_contrast(IMAGE_PATH, area_threshold=0.40)
display(df_cell_4)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)
국소 고대비 영역 (면적 기준 40%),위험 (DANGER),72.53%,65.00%


### [코드 셀 5] 국소 고휘도 영역 판정 알고리즘 — 면적 임계치 25%

In [6]:
@torch.no_grad()
def evaluate_high_luminance(image_path: str, area_threshold: float = 0.25, rel_exceed_ratio: float = 0.30) -> pd.DataFrame:
    """
    Optum 특허(US11769465B1) 기반 상대 비율 고휘도 판정:
    - 프레임 기준 휘도(평균 휘도 L_base) 대비 +30% 이상(L >= L_base * 1.30) 밝고,
      최소 유의 휘도(20 cd/m^2) 이상인 국소 고휘도 픽셀의 면적 비율로 판정합니다.
    """
    rgb = load_single_image_rgb(image_path)
    _, _, lum = srgb_to_linear_and_luminance(rgb)

    l_base = torch.clamp(lum.mean(), min=1.0)
    target_lum = torch.clamp(l_base * (1.0 + rel_exceed_ratio), min=20.0)

    high_lum_mask = lum >= target_lum
    high_lum_area_fraction = float(high_lum_mask.float().mean().item())

    is_danger = high_lum_area_fraction > area_threshold
    verdict = "위험 (DANGER)" if is_danger else "안전 (SAFE)"

    # 방식 1, 2 확률 산출
    rel_ratio_map = torch.clamp((lum - l_base) / torch.clamp(l_base * rel_exceed_ratio, min=1e-6), min=0.0)
    lum_gate = float(torch.clamp(rel_ratio_map[high_lum_mask].mean() if high_lum_mask.any() else rel_ratio_map.max(), 0.0, 1.0).item())
    eta = (high_lum_area_fraction / area_threshold) * lum_gate
    prob_method_1 = (eta ** 2 / (1.0 + eta ** 2)) * 100.0
    prob_method_2 = high_lum_area_fraction * 100.0

    label_pct = int(round(area_threshold * 100))
    return pd.DataFrame([{
        "판정 결과": verdict,
        "판정 근거 확률 - 방식 1 (임계치 달성도 기반)": f"{prob_method_1:.2f}%",
        "판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)": f"{prob_method_2:.2f}%"
    }], index=[f"국소 고휘도 영역 (면적 기준 {label_pct}%)"])

df_cell_5 = evaluate_high_luminance(IMAGE_PATH, area_threshold=0.25)
display(df_cell_5)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)
국소 고휘도 영역 (면적 기준 25%),안전 (SAFE),38.29%,19.69%


### [코드 셀 6] 국소 고휘도 영역 판정 알고리즘 — 면적 임계치 40%

In [7]:
df_cell_6 = evaluate_high_luminance(IMAGE_PATH, area_threshold=0.40)
display(df_cell_6)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)
국소 고휘도 영역 (면적 기준 40%),안전 (SAFE),19.51%,19.69%


### [코드 셀 7] 포화 적색 영역 판정 알고리즘 — 면적 임계치 25%

In [8]:
@torch.no_grad()
def evaluate_saturated_red(image_path: str, area_threshold: float = 0.25) -> pd.DataFrame:
    """
    WCAG 2.2 / ISO 9241-391 포화 적색 기준:
    1) 선형 RGB 기준 R / (R + G + B) >= 0.8 인 포화 적색 픽셀 면적 비율 > area_threshold (25% 또는 40%)
    2) 프레임 내 비적색(또는 대조 색상) 영역과의 CIE 1976 UCS 색도 거리 Delta u'v' > 0.2
    두 조건을 동시에 만족할 때 위험으로 판정합니다.
    """
    rgb = load_single_image_rgb(image_path)
    c_lin, _, _ = srgb_to_linear_and_luminance(rgb)
    r_lin, g_lin, b_lin = c_lin[..., 0], c_lin[..., 1], c_lin[..., 2]

    # 1. 포화 적색 비율 R / (R + G + B) >= 0.8
    rgb_sum = r_lin + g_lin + b_lin
    red_ratio = torch.where(rgb_sum > 1e-4, r_lin / torch.clamp(rgb_sum, min=1e-6), torch.zeros_like(r_lin))
    sat_red_mask = (red_ratio >= 0.8) & (r_lin > 0.01)
    sat_red_area_fraction = float(sat_red_mask.float().mean().item())

    # 2. CIE 1931 XYZ -> CIE 1976 UCS (u', v') 변환 및 색도 차이 Delta u'v' 계산
    x_cie = 0.4124564 * r_lin + 0.3575761 * g_lin + 0.1804375 * b_lin
    y_cie = 0.2126729 * r_lin + 0.7151522 * g_lin + 0.0721750 * b_lin
    z_cie = 0.0193339 * r_lin + 0.1191920 * g_lin + 0.9503041 * b_lin
    denom = x_cie + 15.0 * y_cie + 3.0 * z_cie
    valid_chroma = denom > 1e-4

    # D65 백색점 기본값 (u'_n = 0.1978, v'_n = 0.4683)
    u_prime = torch.where(valid_chroma, (4.0 * x_cie) / torch.clamp(denom, min=1e-6), torch.full_like(x_cie, 0.1978))
    v_prime = torch.where(valid_chroma, (9.0 * y_cie) / torch.clamp(denom, min=1e-6), torch.full_like(y_cie, 0.4683))

    non_red_mask = (~sat_red_mask) & valid_chroma
    if sat_red_mask.any() and non_red_mask.any():
        u_red_mean = u_prime[sat_red_mask].mean()
        v_red_mean = v_prime[sat_red_mask].mean()
        dist_map = torch.sqrt((u_prime[non_red_mask] - u_red_mean) ** 2 + (v_prime[non_red_mask] - v_red_mean) ** 2)
        delta_uv = float(torch.quantile(dist_map, 0.90).item())
    else:
        delta_uv = 0.0

    chroma_satisfied = delta_uv > 0.2
    is_danger = (sat_red_area_fraction > area_threshold) and chroma_satisfied
    verdict = "위험 (DANGER)" if is_danger else "안전 (SAFE)"

    # 방식 1, 2 확률 산출
    red_gate = float(torch.clamp(red_ratio[sat_red_mask].mean() / 0.8 if sat_red_mask.any() else red_ratio.max() / 0.8, 0.0, 1.0).item())
    chroma_gate = min(1.0, delta_uv / 0.2)
    eta = (sat_red_area_fraction / area_threshold) * red_gate * chroma_gate
    prob_method_1 = (eta ** 2 / (1.0 + eta ** 2)) * 100.0
    prob_method_2 = (sat_red_area_fraction * 100.0) if chroma_satisfied else 0.0

    label_pct = int(round(area_threshold * 100))
    return pd.DataFrame([{
        "판정 결과": verdict,
        "판정 근거 확률 - 방식 1 (임계치 달성도 기반)": f"{prob_method_1:.2f}%",
        "판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)": f"{prob_method_2:.2f}%"
    }], index=[f"포화 적색 영역 (면적 기준 {label_pct}%)"])

df_cell_7 = evaluate_saturated_red(IMAGE_PATH, area_threshold=0.25)
display(df_cell_7)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)
포화 적색 영역 (면적 기준 25%),안전 (SAFE),49.37%,24.69%


### [코드 셀 8] 포화 적색 영역 판정 알고리즘 — 면적 임계치 40%

In [9]:
df_cell_8 = evaluate_saturated_red(IMAGE_PATH, area_threshold=0.40)
display(df_cell_8)

,판정 결과,판정 근거 확률 - 방식 1 (임계치 달성도 기반),판정 근거 확률 - 방식 2 (위험 조건 충족 면적 비율)
포화 적색 영역 (면적 기준 40%),안전 (SAFE),27.58%,24.69%
